In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [3]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [4]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [5]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [6]:
words = ["chess", "apartment", "bicycle", "sister"]

In [7]:
glove_wiki_vectors.most_similar(words[0])

[('grandmaster', 0.6360425353050232),
 ('backgammon', 0.6346544027328491),
 ('billiards', 0.6246423721313477),
 ('olympiad', 0.622958779335022),
 ('snooker', 0.6112998723983765),
 ('tennis', 0.6084876656532288),
 ('kasparov', 0.6041906476020813),
 ('scrabble', 0.5954940915107727),
 ('amateur', 0.5934121608734131),
 ('grandmasters', 0.5880254507064819)]

In [8]:
glove_wiki_vectors.most_similar(words[1])

[('bedroom', 0.8140636682510376),
 ('apartments', 0.7545885443687439),
 ('basement', 0.7370947599411011),
 ('hotel', 0.7356935143470764),
 ('rented', 0.7213074564933777),
 ('manhattan', 0.7159244418144226),
 ('room', 0.7060696482658386),
 ('townhouse', 0.7053059935569763),
 ('mansion', 0.70204758644104),
 ('condominium', 0.695074737071991)]

In [9]:
glove_wiki_vectors.most_similar(words[2])

[('bike', 0.8968486189842224),
 ('motorcycle', 0.7976688146591187),
 ('motorbike', 0.7346670627593994),
 ('bicycles', 0.7253736257553101),
 ('bikes', 0.7156639099121094),
 ('riding', 0.7103949189186096),
 ('car', 0.686353862285614),
 ('ride', 0.6754668354988098),
 ('rides', 0.6725955605506897),
 ('trolley', 0.6721103191375732)]

In [10]:
glove_wiki_vectors.most_similar(words[3])

[('daughter', 0.8687513470649719),
 ('mother', 0.8647425174713135),
 ('wife', 0.8372183442115784),
 ('daughters', 0.7978368401527405),
 ('niece', 0.7832933068275452),
 ('sisters', 0.7734189033508301),
 ('aunt', 0.7558451890945435),
 ('cousin', 0.755153477191925),
 ('husband', 0.7548229694366455),
 ('father', 0.754813551902771)]

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [11]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [12]:
for x in word_pairs:
    print("Similarity between {} and {}: {}".format(x[0], x[1], glove_wiki_vectors.similarity(x[0], x[1])))

Similarity between coast and shore: 0.7000271677970886
Similarity between clothes and closet: 0.5462759733200073
Similarity between old and new: 0.6432487964630127
Similarity between smart and intelligent: 0.7552732825279236
Similarity between dog and cat: 0.8798075318336487
Similarity between tree and lawyer: 0.07671944797039032


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [13]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [14]:
for x in test_words:
    print("{} in glove_wiki_vectors: {}".format(x, x in glove_wiki_vectors))

covididiot in glove_wiki_vectors: False
fomo in glove_wiki_vectors: False
frenemies in glove_wiki_vectors: True
anthropause in glove_wiki_vectors: False
photobomb in glove_wiki_vectors: False
selfie in glove_wiki_vectors: False
pxg in glove_wiki_vectors: False
pacg in glove_wiki_vectors: False
cct in glove_wiki_vectors: True
escc in glove_wiki_vectors: True


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [15]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [16]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [17]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283238)

In [18]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.351824)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [19]:
glove_wiki_vectors.similarity("black","crime")

np.float32(0.43576178)

In [20]:
glove_wiki_vectors.similarity("white","crime")

np.float32(0.38129756)

In [21]:
analogy("man","smart","woman")

man : smart :: woman : ?


,Analogy word,Score
0,intelligent,0.654885
1,sexy,0.597897
2,sophisticated,0.574362
3,mom,0.557256
4,cute,0.549813
5,kids,0.540299
6,pretty,0.530862
7,savvy,0.530233
8,innovative,0.530197
9,attractive,0.528517


In [22]:
glove_wiki_vectors.similarity("success","pretty")

np.float32(0.48183727)

In [23]:
glove_wiki_vectors.similarity("success","smart")

np.float32(0.33908477)

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

1. There are certainly some worrisome biases. A clear one is the apparent association of the word "black" and "crime" compared to "white" and "crime". This bias could subconsciously teach a model to be racist when presented with the concept of white and black individuals. Furthermore, the analogy between "man", "smart" and "woman" lists "sexy" as the second highest analogous word. This is very misogionistic and is teaching the model that women being "sexy" is almost as important as a man being "smart". 
2. Evidently these biases can be very harming the real world. For example, if machine learning was used to filter job applications due to biases towards "traditional" male positions a model might learn that if an applicant is female they shouldn't be recommended based on its biased past training data.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [24]:
from sklearn.datasets import fetch_20newsgroups

In [25]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

,text,target,target_name
0,"You know, I was reading 18 U.S.C. 922 and some...",6,talk.politics.guns
1,\n\n\nIt's not a bad question: I don't have an...,1,comp.graphics
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
...,...,...,...
4558,Hi Everyone ::\n\nI am looking for some soft...,1,comp.graphics
4559,Archive-name: x-faq/part3\nLast-modified: 1993...,2,comp.windows.x
4560,"\nThat's nice, but it doesn't answer the quest...",6,talk.politics.guns
4561,"Hi,\n I just got myself a Gateway 4DX-33V ...",2,comp.windows.x


In [26]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [27]:
import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [28]:
import re
import spacy

nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

for word in {"edu", "com", "writes", "write", "article", "organization", "nntp", "host", "posting", "post"}:
    nlp.vocab[word].is_stop = True

url_re = re.compile(r"(https?://\\S+|www\\.\\S+)")
email_re = re.compile(r"\\b[\\w.-]+@[\\w.-]+\\.\\w+\\b")
num_re = re.compile(r"\\b\\d+(?:\\.\\d+)?\\b")
artifact_re = re.compile(r"[_`~><*\\[\\]()|]")
ws_re = re.compile(r"\\s+")
allowed_pos = {"NOUN", "PROPN", "ADJ", "VERB"}
placeholder_tokens = {"url", "email", "num"}

def normalize_text(text):
    text = text.lower()
    text = url_re.sub(" URL ", text)
    text = email_re.sub(" EMAIL ", text)
    text = num_re.sub(" NUM ", text)
    text = text.replace("\\n", " ").replace("\\t", " ")
    text = artifact_re.sub(" ", text)
    return ws_re.sub(" ", text).strip()

def preprocess_doc(doc):
    tokens = []
    for token in doc:
        lemma = token.lemma_.lower().strip()

        if token.like_url or token.like_email or lemma in placeholder_tokens:
            continue
        if token.is_space or token.is_punct or token.is_stop:
            continue
        if lemma in {"", "-pron-"}:
            continue
        if token.pos_ not in allowed_pos:
            continue
        if token.is_digit or any(char.isdigit() for char in lemma):
            continue
        if not lemma.isalpha():
            continue
        if len(lemma) < 3:
            continue

        tokens.append(lemma)

    return " ".join(tokens)

clean_text = [normalize_text(text) for text in df["text"].fillna("")]
df["text_pp"] = [preprocess_doc(doc) for doc in nlp.pipe(clean_text, batch_size=128)]
df["text_pp"] = df["text_pp"].str.replace(r"\\s+", " ", regex=True).str.strip()

In [29]:
...

Ellipsis

In [30]:
...

Ellipsis

In [38]:
df[["target_name", "text_pp"]].iloc[2:25]

,target_name,text_pp
2,comp.graphics,actuallay hand support idea have newsgroup asp...
3,comp.windows.x,follow problem bug appreciate help create wind...
4,talk.politics.mideast,late upi foreign ministry spokesman ferhat ata...
5,soc.religion.christian,like subscribe leadership magazine wonder disk...
6,talk.politics.mideast,simple soviet armenian government pay crime ge...
7,rec.sport.hockey,squid tradition alive fish unh game
8,talk.politics.mideast,center policy research cpr subject question is...
9,comp.windows.x,openlook window manager source available mit c...
10,talk.politics.guns,question assert suitable real hunting problem ...
11,comp.windows.x,motif mailing list locate like add delete list...


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [32]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=8, random_state=42, learning_method="batch", max_iter=10)

vec = CountVectorizer(stop_words="english")
X = vec.fit_transform(df["text_pp"])

lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",8
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [33]:
lda.components_

array([[ 0.12507319,  0.125     ,  0.12552344, ...,  2.12481628,
         0.125     ,  0.12500001],
       [15.31932485,  0.12512402,  0.12500002, ...,  0.12503754,
         0.12500001,  0.12500001],
       [ 0.12509689,  0.125     ,  0.12854259, ...,  0.125     ,
         0.125     ,  1.12391781],
       ...,
       [ 0.12501322,  0.12503048,  1.12040276, ...,  0.12514617,
         0.125     ,  0.12500001],
       [13.93033106,  0.12599763,  0.12500002, ...,  0.125     ,
         0.12545139,  0.12598044],
       [ 0.125     ,  0.125     ,  0.12500002, ...,  0.125     ,
         0.12500001,  0.12500001]], shape=(8, 28402))

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

In [34]:
import mglearn

mglearn.tools.print_topics(
    topics=range(8),
    feature_names=vec.get_feature_names_out(),
    n_words=10,
    sorting=np.argsort(lda.components_, axis=1)[:, ::-1]
)

topic 0       topic 1       topic 2       topic 3       topic 4       topic 5       
--------      --------      --------      --------      --------      --------      
know          player        say           gun           god           file          
book          entry         people        people        believe       window        
think         team          know          right         think         program       
time          jpeg          come          law           people        use           
read          hockey        time          weapon        jesus         run           
people        good          think         firearm       christian     server        
image         league        tell          state         church        display       
good          year          kill          government    know          widget        
send          nhl           woman         control       say           available     
thing         new           happen        crime         thing    

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [35]:
doc_topic = lda.transform(X)

for i in range(5):
    assigned_topic = doc_topic[i].argmax()
    print(f"Document {i}: Topic {assigned_topic}")
    print(df['text'].iloc[i][:100])

Document 0: Topic 3
You know, I was reading 18 U.S.C. 922 and something just did not make 
sence and I was wondering if 
Document 1: Topic 5



It's not a bad question: I don't have any refs that list this algorithm
either. But thinking abou
Document 2: Topic 0

Actuallay I don't, but on the other hand I don't support the idea of having
one newsgroup for every
Document 3: Topic 5
The following problem is really bugging me,
and I would appreciate any help.

I create two windows:

Document 4: Topic 7


  This is the latest from UPI 

     Foreign Ministry spokesman Ferhat Ataman told journalists Tur


<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. We obtain information about the content that is being "rated" by users and create features from said information. From there we create a user profile per user and treat the predictions as a set of regression problems. Each user has a regression model fit upon their ratings data. As the user adds more reviews the model is updated to provide more accurate ratings.
2. Recommender systems can contriubte to reinforcement bias/filter bubbles by further suggesting malicious or hateful content to users just because they viewed a couple pieces of content related to a topic. (ex: manosphere content on youtube/social media) Limited discovery of non-mainstream content. Recommender systems are designed to show content that a user "should" like or if the user hasn't generated a unique profile yet then it will show the most generally popular content. This can make non-mainstream content fall unnoticed.
3. The model reads words in parallel and uses "attention" to determine which words are of most importance and what to focus on. 

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)